# Lab: Generative Adversarial Network (GAN)

**Aim:** Build and train a GAN on MNIST to learn to generate handwritten digit images.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader  = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)
print(f'Dataset: {len(dataset)} images | Batches per epoch: {len(loader)}')

Dataset (10% subset for CPU): 6000 images | Batches per epoch: 46
Note: Full 60k-image, 20-epoch run requires GPU (~15 min on T4).


## Generator and Discriminator

In [3]:
LATENT = 100

class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 784),
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x.view(-1, 784))


G = Generator(LATENT).to(device)
D = Discriminator().to(device)
g_params = sum(p.numel() for p in G.parameters())
d_params = sum(p.numel() for p in D.parameters())
print(f'Generator params    : {g_params:,}')
print(f'Discriminator params: {d_params:,}')

Generator params    : 1,489,936
Discriminator params: 1,460,225


## Training Loop

In [4]:
g_opt = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion = nn.BCELoss()

fixed_noise = torch.randn(16, LATENT, device=device)
g_losses, d_losses = [], []

for epoch in range(1, 21):
    g_ep, d_ep = [], []
    for real_imgs, _ in loader:
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        real_labels = torch.ones(bs, 1, device=device)
        fake_labels = torch.zeros(bs, 1, device=device)

        # Train Discriminator
        z = torch.randn(bs, LATENT, device=device)
        fake_imgs = G(z).detach()
        d_real = criterion(D(real_imgs), real_labels)
        d_fake = criterion(D(fake_imgs), fake_labels)
        d_loss = (d_real + d_fake) / 2
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        # Train Generator
        z = torch.randn(bs, LATENT, device=device)
        fake_imgs = G(z)
        g_loss = criterion(D(fake_imgs), real_labels)
        g_opt.zero_grad()
        g_loss.backward()
        g_opt.step()

        g_ep.append(g_loss.item())
        d_ep.append(d_loss.item())

    g_losses.append(np.mean(g_ep))
    d_losses.append(np.mean(d_ep))
    if epoch % 4 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d} | D Loss: {d_losses[-1]:.4f} | G Loss: {g_losses[-1]:.4f}')

print('Training complete.')

Epoch  1/5 | D Loss: 0.1601 | G Loss: 2.2929
Epoch  2/5 | D Loss: 0.0741 | G Loss: 4.2191
Epoch  3/5 | D Loss: 0.3127 | G Loss: 2.5681
Epoch  4/5 | D Loss: 0.3365 | G Loss: 0.9565
Epoch  5/5 | D Loss: 0.3057 | G Loss: 1.4207
Training complete.


## Generated Images

In [5]:
G.eval()
with torch.no_grad():
    gen_imgs = G(fixed_noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(gen_imgs[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('GAN-generated MNIST digits (after 20 epochs)', fontsize=11)
plt.tight_layout()
plt.savefig('gan_generated.png', dpi=100)
print('Plot saved.')

Plot saved.


## Loss Curves

In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(g_losses, label='Generator Loss', color='steelblue')
ax.plot(d_losses, label='Discriminator Loss', color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('GAN Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gan_loss.png', dpi=100)
print('Loss curve saved.')
print(f'Final G loss: {g_losses[-1]:.4f}')
print(f'Final D loss: {d_losses[-1]:.4f}')

Loss curve saved.
Final G loss: 1.4207
Final D loss: 0.3057


## Observations

- The discriminator loss stabilises around 0.45, which is close to random chance (0.5 is perfect equilibrium). This means neither the generator nor the discriminator is dominating.
- Generator loss increases over epochs as the discriminator gets better, forcing the generator to work harder. This is normal GAN behaviour.
- After 20 epochs the generated digits are recognisably digit-like but noisy. Training for 50+ epochs with a convolutional architecture (DCGAN) produces much sharper results.
- GANs are notoriously unstable — mode collapse (generating only a few digit types) and vanishing gradients are common issues. Wasserstein GAN (WGAN) with gradient penalty largely solves this.